# Job Market Intelligence – Data Cleaning

This notebook cleans and prepares the No Fluff Jobs dataset for exploratory data analysis and machine learning.

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "nofluff_it_jobs.csv"
)

df = pd.read_csv(RAW_PATH)

raw_row_count = len(df)

expired_offer_mask = (
    df["company"]
    .fillna("")
    .str.strip()
    .eq("Zobacz profil firmy")
)

df = (
    df.loc[~expired_offer_mask]
    .copy()
    .reset_index(drop=True)
)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Raw rows:", raw_row_count)
print("Expired offers removed:", expired_offer_mask.sum())
print("Dataset shape after removal:", df.shape)

PROJECT_ROOT: G:\pandas\job_market_intelligence
Raw rows: 3286
Expired offers removed: 430
Dataset shape after removal: (2856, 25)


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2856 entries, 0 to 2855
Data columns (total 25 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   job_id                2856 non-null   object 
 1   url                   2856 non-null   object 
 2   title                 2851 non-null   object 
 3   category              2638 non-null   object 
 4   experience            2762 non-null   object 
 5   experience_years_min  1630 non-null   float64
 6   workplace             2533 non-null   object 
 7   job_locations         1769 non-null   object 
 8   company               2856 non-null   object 
 9   company_size          2022 non-null   object 
 10  company_founded       2022 non-null   float64
 11  company_locations     2022 non-null   object 
 12  salary_min            2622 non-null   float64
 13  salary_max            2622 non-null   float64
 14  salary_currency       2622 non-null   object 
 15  salary_period        

## 1. Missing values

Check the number and percentage of missing values in each column.

In [3]:
missing_values = (
    df.isna()
    .sum()
    .to_frame("missing_count")
)

missing_values["missing_percent"] = (
    missing_values["missing_count"]
    / len(df)
    * 100
).round(2)

missing_values = missing_values.sort_values(
    "missing_percent",
    ascending=False
)

missing_values

,missing_count,missing_percent
salary_period,1271,44.50
experience_years_min,1226,42.93
job_locations,1087,38.06
nice_to_have,1065,37.29
company_locations,834,29.20
company_size,834,29.20
company_founded,834,29.20
contract_type,712,24.93
workplace,323,11.31
responsibilities,249,8.72


## 2. Duplicates

Check for duplicated rows, job IDs, and job URLs.

In [4]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate job IDs:", df["job_id"].duplicated().sum())
print("Duplicate URLs:", df["url"].duplicated().sum())

Duplicate rows: 0
Duplicate job IDs: 0
Duplicate URLs: 0


## 3. Categorical values

Inspect the most important categorical columns and identify inconsistent or unexpected values.

In [5]:
columns_to_check = [
    "experience",
    "workplace",
    "category",
    "contract_type",
    "salary_currency",
]

for col in columns_to_check:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(30))


--- experience ---
experience
Senior    1429
Mid       1095
Junior     156
NaN         94
Expert      82
Name: count, dtype: int64

--- workplace ---
workplace
Hybrid    1532
Remote     991
NaN        323
Onsite      10
Name: count, dtype: int64

--- category ---
category
Backend               252
NaN                   218
Data                  211
DevOps                208
Testing               175
Java, Backend         171
ERP                   169
Project Manager       159
Security              147
Architecture          126
AI                    118
Fullstack              88
Support                81
Product Management     69
Python, Data           69
.NET, Backend          61
Business Analysis      57
Java, Fullstack        54
Frontend               53
Python, Backend        52
Python, AI             45
Mobile                 43
.NET, Fullstack        40
Embedded               38
Python, Testing        28
Python, DevOps         26
Python, Fullstack      22
Java, Testing          1

## 4. Numerical values

Inspect numerical columns and check for invalid or unexpected values.

In [6]:
numerical_columns = [
    "experience_years_min",
    "company_founded",
    "salary_min",
    "salary_max",
]

df[numerical_columns].describe()

,experience_years_min,company_founded,salary_min,salary_max
count,1630.000000,2022.000000,2622.000000,2622.000000
mean,4.752761,1999.959941,15160.276888,19785.908085
std,1.995293,32.193546,29771.884151,41715.029367
min,1.000000,1845.000000,28.000000,32.000000
25%,3.000000,1996.000000,170.000000,200.000000
50%,5.000000,2007.000000,12000.000000,16500.000000
75%,5.000000,2015.000000,22400.000000,28000.000000
max,15.000000,2026.000000,350700.000000,474400.000000


In [7]:
print(
    "Salary min > salary max:",
    (df["salary_min"] > df["salary_max"]).sum()
)

print(
    "Salary <= 0:",
    (
        (df["salary_min"] <= 0)
        | (df["salary_max"] <= 0)
    ).sum()
)

print(
    "Company founded > 2026:",
    (df["company_founded"] > 2026).sum()
)

print(
    "Experience years < 0:",
    (df["experience_years_min"] < 0).sum()
)

Salary min > salary max: 0
Salary <= 0: 0
Company founded > 2026: 0
Experience years < 0: 0


In [8]:
low_salary_jobs = df[
    (df["salary_min"] < 5000)
    | (df["salary_max"] < 5000)
][
    [
        "title",
        "company",
        "salary_min",
        "salary_max",
        "salary_currency",
        "contract_type",
        "url",
    ]
]

print("Low salary jobs:", len(low_salary_jobs))

low_salary_jobs

Low salary jobs: 1118


,title,company,salary_min,salary_max,salary_currency,contract_type,url
4,Administrator Systemów Linux/BSD (z elementami...,Fudo Security,72.0,96.0,PLN,B2B,https://nofluffjobs.com/pl/job/administrator-s...
6,Administrator Windows (transformacja),Mindbox Sp. z o.o.,800.0,1000.0,PLN,"B2B, Umowa o pracę",https://nofluffjobs.com/pl/job/administrator-w...
7,Adobe Analytics Specialist - IT Contracting,Michael Page,110.0,135.0,PLN,B2B,https://nofluffjobs.com/pl/job/adobe-analytics...
14,Agile Delivery Manager,Link Group,145.0,160.0,PLN,B2B,https://nofluffjobs.com/pl/job/agile-delivery-...
16,AI Agent Developer (Microsoft Copilot Studio),Upvanta,150.0,200.0,PLN,B2B,https://nofluffjobs.com/pl/job/ai-agent-develo...
...,...,...,...,...,...,...,...
2835,Vulnerability Response Senior SME,Mindbox Sp. z o.o.,1400.0,1550.0,PLN,"B2B, Umowa o pracę",https://nofluffjobs.com/pl/job/vulnerability-r...
2839,Windows Support Engineer,Ework Group,32.0,35.0,EUR,B2B,https://nofluffjobs.com/pl/job/windows-support...
2846,Workfront Specialist,Square One Resources,200.0,240.0,PLN,B2B,https://nofluffjobs.com/pl/job/workfront-speci...
2847,XSLT/.NET Developer (he/she),ASTEK Polska,120.0,150.0,PLN,B2B,https://nofluffjobs.com/pl/job/xslt-net-develo...


In [9]:
df[["valid_until", "start_date", "scraped_at"]].head(20)

,valid_until,start_date,scraped_at
0,11.09.2026,ASAP,2026-08-23 01:15:35
1,06.09.2026,ASAP,2026-08-23 01:15:39
2,NaN,ASAP,2026-08-23 01:15:43
3,03.09.2026,ASAP,2026-08-23 01:15:46
4,31.08.2026,ASAP,2026-08-23 01:15:50
5,28.08.2026,ASAP,2026-08-23 01:15:54
6,15.09.2026,NaN,2026-08-23 01:15:58
7,29.08.2026,ASAP,2026-08-23 01:16:02
8,09.09.2026,ASAP,2026-08-23 01:16:05
9,09.09.2026,ASAP,2026-08-23 01:16:09


In [10]:
start_date_check = (
    df["start_date"]
    .dropna()
    .astype(str)
    .str.len()
)

start_date_check.describe()

count    2621.000000
mean        4.510492
std         1.674342
min         4.000000
25%         4.000000
50%         4.000000
75%         4.000000
max        10.000000
Name: start_date, dtype: float64

In [11]:
df.loc[
    df["start_date"]
    .fillna("")
    .astype(str)
    .str.len()
    .sort_values(ascending=False)
    .head(20)
    .index,
    ["title", "start_date"]
]

,title,start_date
2594,Solution Architect,2026-08-10
2819,UX/UI Designer,2026-06-16
2603,Solutions Architect,2026-09-01
2608,"Specialist Quantitative Developer (Python, Pan...",2026-07-01
2569,Software Solution Architect - Warehouse Softwa...,2026-07-15
2548,Software Engineer Performance,2026-06-17
2561,Software Factory Architect,2026-09-01
2562,Software Factory Architect (SOFA ARCH),2026-09-01
2487,Software Backend Developer / Senior Specialist,2026-08-01
2470,Snowflake Data Engineer,2026-09-01


In [12]:
invalid_start_dates = df[
    df["start_date"].notna()
    & (df["start_date"] != "ASAP")
    & ~df["start_date"].str.match(
        r"^\d{4}-\d{2}-\d{2}$",
        na=False
    )
]

print("Unexpected start_date values:", len(invalid_start_dates))

invalid_start_dates[
    ["title", "start_date"]
].head(20)

Unexpected start_date values: 0


,title,start_date


## 5. Date columns

Convert date-related columns to datetime format and preserve the ASAP information separately.

In [13]:
df["valid_until"] = pd.to_datetime(
    df["valid_until"],
    format="%d.%m.%Y",
    errors="coerce"
)

df["scraped_at"] = pd.to_datetime(
    df["scraped_at"],
    errors="coerce"
)

df["start_asap"] = (
    df["start_date"]
    .eq("ASAP")
)

df["start_date_parsed"] = pd.to_datetime(
    df["start_date"].where(
        df["start_date"] != "ASAP"
    ),
    format="%Y-%m-%d",
    errors="coerce"
)

df[
    [
        "start_date",
        "start_asap",
        "start_date_parsed",
        "valid_until",
        "scraped_at",
    ]
].head(20)

,start_date,start_asap,start_date_parsed,valid_until,scraped_at
0,ASAP,True,NaT,2026-09-11,2026-08-23 01:15:35
1,ASAP,True,NaT,2026-09-06,2026-08-23 01:15:39
2,ASAP,True,NaT,NaT,2026-08-23 01:15:43
3,ASAP,True,NaT,2026-09-03,2026-08-23 01:15:46
4,ASAP,True,NaT,2026-08-31,2026-08-23 01:15:50
5,ASAP,True,NaT,2026-08-28,2026-08-23 01:15:54
6,NaN,False,NaT,2026-09-15,2026-08-23 01:15:58
7,ASAP,True,NaT,2026-08-29,2026-08-23 01:16:02
8,ASAP,True,NaT,2026-09-09,2026-08-23 01:16:05
9,ASAP,True,NaT,2026-09-09,2026-08-23 01:16:09


In [14]:
print("valid_until NaT:", df["valid_until"].isna().sum())
print("scraped_at NaT:", df["scraped_at"].isna().sum())

print()
print("ASAP:", df["start_asap"].sum())
print("Parsed start dates:", df["start_date_parsed"].notna().sum())
print("Missing original start_date:", df["start_date"].isna().sum())

print()
print(
    "Total start_date:",
    df["start_asap"].sum()
    + df["start_date_parsed"].notna().sum()
    + df["start_date"].isna().sum()
)

valid_until NaT: 227
scraped_at NaT: 0

ASAP: 2398
Parsed start dates: 223
Missing original start_date: 235

Total start_date: 2856


In [15]:
df["company_size"].value_counts(dropna=False).head(30)

company_size
NaN          834
500 - 999    329
1000+        319
250 - 499    251
100+         221
50 - 249     129
40+           90
+1000         79
500           60
1500          49
10 - 49       44
+6000         42
350+          31
250           29
2000+         28
500+          22
300+          20
1700+         19
7500+         15
+2400         15
200+          14
2400+         14
450+          13
110000        12
5000          12
50+           12
13400+        12
60+           10
20+            9
600+           9
Name: count, dtype: int64

In [16]:
def company_size_format(value):

    if pd.isna(value):
        return "missing"

    value = str(value).strip()
    value = value.replace(",", "")
    value = value.replace(" ", "")

    if re.fullmatch(r"\d+\s*-\s*\d+", value):
        return "range"

    if re.fullmatch(r"\d+\+", value):
        return "min_plus"

    if re.fullmatch(r"\+\d+", value):
        return "plus_min"

    if re.fullmatch(r">\d+", value):
        return "greater_than"

    if re.fullmatch(r"\d+", value):
        return "exact"

    return "other"


df["company_size"].apply(
    company_size_format
).value_counts()

company_size
min_plus        910
missing         834
range           766
exact           197
plus_min        143
greater_than      6
Name: count, dtype: int64

In [17]:
other_company_sizes = df[
    df["company_size"].apply(company_size_format) == "other"
]["company_size"].value_counts()

other_company_sizes

Series([], Name: count, dtype: int64)

## 6. Company size

Standardize company size values, extract employee-count bounds, and create consistent company-size segments. Open-ended values are assigned using their reported lower bound.

In [18]:
def parse_company_size(value):

    if pd.isna(value):
        return np.nan, np.nan

    value = str(value).strip()

    value = value.replace(",", "")
    value = value.replace(" ", "")

    match = re.fullmatch(r"(\d+)-(\d+)", value)

    if match:
        return (
            int(match.group(1)),
            int(match.group(2)),
        )

    match = re.fullmatch(r"(\d+)\+", value)

    if match:
        return int(match.group(1)), np.nan

    match = re.fullmatch(r"\+(\d+)", value)

    if match:
        return int(match.group(1)), np.nan

    match = re.fullmatch(r">(\d+)", value)

    if match:
        return int(match.group(1)) + 1, np.nan

    if re.fullmatch(r"\d+", value):
        number = int(value)
        return number, number

    return np.nan, np.nan

In [19]:
df[
    ["company_size_min", "company_size_max"]
] = df["company_size"].apply(
    lambda x: pd.Series(
        parse_company_size(x)
    )
)

df[
    [
        "company_size",
        "company_size_min",
        "company_size_max",
    ]
].head(20)

,company_size,company_size_min,company_size_max
0,200+,200.0,NaN
1,5000,5000.0,5000.0
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,50 - 249,50.0,249.0
5,50-100,50.0,100.0
6,NaN,NaN,NaN
7,NaN,NaN,NaN
8,1000+,1000.0,NaN
9,1000+,1000.0,NaN


In [20]:
unparsed_company_sizes = df.loc[
    df["company_size"].notna()
    & df["company_size_min"].isna(),
    "company_size",
].value_counts()

unparsed_company_sizes

Series([], Name: count, dtype: int64)

In [21]:
print(
    "Original company size available:",
    df["company_size"].notna().sum()
)

print(
    "Parsed company size:",
    df["company_size_min"].notna().sum()
)

Original company size available: 2022
Parsed company size: 2022


In [22]:
company_size_order = [
    "Small",
    "Medium",
    "Large",
    "Enterprise",
    "Missing",
]

df["company_size_segment"] = pd.cut(
    df["company_size_min"],
    bins=[0, 50, 250, 1000, np.inf],
    labels=company_size_order[:-1],
    right=False,
)

df["company_size_segment"] = (
    df["company_size_segment"]
    .cat.add_categories(["Missing"])
    .fillna("Missing")
)

df["company_size_segment"].value_counts(
    sort=False
)

company_size_segment
Small         154
Medium        432
Large         776
Enterprise    660
Missing       834
Name: count, dtype: int64

## 7. Numerical data types

Convert integer-like numerical columns to nullable integer format.

In [23]:
df["experience_years_min"] = (
    df["experience_years_min"]
    .astype("Int64")
)

df["company_founded"] = (
    df["company_founded"]
    .astype("Int64")
)

df["company_size_min"] = (
    df["company_size_min"]
    .astype("Int64")
)

df["company_size_max"] = (
    df["company_size_max"]
    .astype("Int64")
)

df[
    [
        "experience_years_min",
        "company_founded",
        "company_size_min",
        "company_size_max",
    ]
].dtypes

experience_years_min    Int64
company_founded         Int64
company_size_min        Int64
company_size_max        Int64
dtype: object

## 8. Text standardization

Remove leading and trailing whitespace and convert empty strings to missing values.

In [24]:
text_columns = df.select_dtypes(
    include="object"
).columns

for col in text_columns:
    df[col] = (
        df[col]
        .str.strip()
        .replace("", pd.NA)
    )

print("Text columns cleaned:", len(text_columns))

Text columns cleaned: 19


In [25]:
empty_strings = (
    df[text_columns]
    .eq("")
    .sum()
    .sum()
)

print("Empty strings remaining:", empty_strings)

Empty strings remaining: 0


## 9. Salary analysis flag

Preserve the original salary values and create a conservative flag for salary records suitable for monthly PLN analysis.

In [26]:
df["salary_analysis_eligible"] = (
    df["salary_min"].notna()
    & df["salary_max"].notna()
    & (df["salary_currency"] == "PLN")
    & (df["salary_min"] >= 5000)
    & (df["salary_max"] >= 5000)
)

print(
    df["salary_analysis_eligible"]
    .value_counts()
)

print(
    "\nEligible salary offers:",
    df["salary_analysis_eligible"].sum()
)

salary_analysis_eligible
True     1470
False    1386
Name: count, dtype: int64

Eligible salary offers: 1470


## 10. Final numerical types

Convert salary columns to nullable integer format while preserving missing values.

In [27]:
df["salary_min"] = df["salary_min"].astype("Int64")
df["salary_max"] = df["salary_max"].astype("Int64")

df[
    [
        "salary_min",
        "salary_max",
        "experience_years_min",
        "company_founded",
        "company_size_min",
        "company_size_max",
    ]
].dtypes

salary_min              Int64
salary_max              Int64
experience_years_min    Int64
company_founded         Int64
company_size_min        Int64
company_size_max        Int64
dtype: object

In [28]:
PROCESSED_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLEAN_PATH = (
    PROCESSED_DIR
    / "nofluff_it_jobs_clean.csv"
)

In [29]:
import re
import pandas as pd

def clean_experience(row):

    title = str(row["title"]).lower()

    levels_found = []

    if re.search(r"\b(junior|jr)\b", title):
        levels_found.append("Junior")

    if re.search(r"\b(mid|regular)\b", title):
        levels_found.append("Mid")

    if re.search(r"\b(senior|sr)\b", title):
        levels_found.append("Senior")

    if re.search(r"\b(expert|principal|head)\b", title):
        levels_found.append("Expert")

    if len(set(levels_found)) == 1:
        return levels_found[0]

    return row["experience"]


df["experience_clean"] = df.apply(
    clean_experience,
    axis=1
)

In [30]:
changed_mask = (
    df["experience"].fillna("Missing")
    !=
    df["experience_clean"].fillna("Missing")
)

print(
    "Changed experience levels:",
    changed_mask.sum()
)

Changed experience levels: 122


In [31]:
df["experience_clean"].value_counts(
    dropna=False
)

experience_clean
Senior    1485
Mid       1040
Junior     121
Expert     118
NaN         92
Name: count, dtype: int64

In [32]:
def extract_experience_years_v3(text):

    if pd.isna(text):
        return pd.NA

    text = str(text).lower()

    patterns = [
  
        (
            r"(?:minimum|min\.?|co najmniej|od)?\s*"
            r"(\d{1,2})"
            r"\s*"
            r"(?:(?:[-–]|do)\s*\d{1,2}\+?)?"
            r"\s*\+?"
            r"\s*"
            r"(?:lat|lata|rok|roku)"
            r"[^.!?\n]{0,50}"
            r"doświadczen"
        ),


        (
            r"(?:minimum|min\.?|at least)?\s*"
            r"(\d{1,2})"
            r"\s*"
            r"(?:(?:[-–]|to)\s*\d{1,2}\+?)?"
            r"\s*\+?"
            r"\s*"
            r"(?:years?|yrs?)"
            r"[^.!?\n]{0,50}"
            r"experience"
        ),

        (
            r"experience"
            r"[^.!?\n]{0,30}"
            r"(\d{1,2})"
            r"\s*\+?"
            r"\s*(?:years?|yrs?)"
        ),
    ]

    for pattern in patterns:
        match = re.search(
            pattern,
            text,
            re.IGNORECASE
        )

        if match:
            return int(match.group(1))

    return pd.NA

In [33]:
df["experience_years_v3"] = (
    df["requirements"]
    .apply(extract_experience_years_v3)
    .astype("Int64")
)

In [34]:
df["experience_years_confident"] = (
    df["experience_years_min"]
    .where(
        df["experience_years_v3"].notna()
        & (
            df["experience_years_min"]
            == df["experience_years_v3"]
        )
    )
    .astype("Int64")
)

In [35]:
df = df.drop(
    columns=["experience_years_v3"]
)

In [36]:
df = df.drop(columns=["experience"])

df = df.rename(
    columns={
        "experience_clean": "experience"
    }
)

In [37]:
df["experience"].value_counts(dropna=False)

experience
Senior    1485
Mid       1040
Junior     121
Expert     118
NaN         92
Name: count, dtype: int64

In [38]:
df = df.drop(
    columns=[
        "experience_years_min",
    ],
    errors="ignore"
)

In [39]:
df = df.rename(
    columns={
        "experience_years_confident": "experience_years_min"
    }
)

In [40]:
print("Dataset shape:", df.shape)

print()
print(df["experience"].value_counts(dropna=False))

print()
print(
    "Experience years available:",
    df["experience_years_min"].notna().sum()
)

print(
    "Duplicate URLs:",
    df["url"].duplicated().sum()
)

print(
    "Missing titles:",
    df["title"].isna().sum()
)

Dataset shape: (2856, 31)

experience
Senior    1485
Mid       1040
Junior     121
Expert     118
NaN         92
Name: count, dtype: int64

Experience years available: 1437
Duplicate URLs: 0
Missing titles: 5


In [41]:
df.to_csv(
    CLEAN_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", CLEAN_PATH)

Saved: G:\pandas\job_market_intelligence\data\processed\nofluff_it_jobs_clean.csv


In [42]:
df.head()

,job_id,url,title,category,workplace,job_locations,company,company_size,company_founded,company_locations,...,start_date,scraped_at,start_asap,start_date_parsed,company_size_min,company_size_max,company_size_segment,salary_analysis_eligible,experience,experience_years_min
0,administrator-it-iteo-katowice,https://nofluffjobs.com/pl/job/administrator-i...,Administrator IT,NaN,NaN,Katowice,iteo S.A.,200+,2011,Katowice + 4,...,ASAP,2026-08-23 01:15:35,True,NaT,200,<NA>,Medium,True,Mid,<NA>
1,administrator-ka-aplikacji-biznesowych-departa...,https://nofluffjobs.com/pl/job/administrator-k...,Administrator/-ka Aplikacji Biznesowych - Depa...,Support,Hybrid,Warszawa,T-Mobile Polska,5000,1996,Warszawa,...,ASAP,2026-08-23 01:15:39,True,NaT,5000,5000,Enterprise,True,Mid,<NA>
2,administrator-ka-it-ima-polska-spolka-akcyjna-...,https://nofluffjobs.com/pl/job/administrator-k...,Administrator/ka IT,NaN,Onsite,NaN,IMA POLSKA SPÓŁKA AKCYJNA,NaN,<NA>,NaN,...,ASAP,2026-08-23 01:15:43,True,NaT,<NA>,<NA>,Missing,True,Mid,<NA>
3,administrator-sap-basis-k-m-modivo-platform-po...,https://nofluffjobs.com/pl/job/administrator-s...,ADMINISTRATOR SAP BASIS (k/m),ERP,Onsite,NaN,MODIVO PLATFORM,NaN,<NA>,NaN,...,ASAP,2026-08-23 01:15:46,True,NaT,<NA>,<NA>,Missing,True,Mid,<NA>
4,administrator-systemow-linux-bsd-z-elementami-...,https://nofluffjobs.com/pl/job/administrator-s...,Administrator Systemów Linux/BSD (z elementami...,NaN,Hybrid,Warszawa,Fudo Security,50 - 249,2004,Warsaw,...,ASAP,2026-08-23 01:15:50,True,NaT,50,249,Medium,False,Mid,3
